# `quant-research-framework` walkthrough

This notebook is a 10-minute tour of the engine. It:

1. Generates a small synthetic OHLC CSV (no network).
2. Imports the package.
3. Runs the default IS / OOS / WFO + robustness pipeline.
4. Plots the equity curve from the trade ledger.
5. Toggles `USE_REGIME_SEG = True` and re-runs to show the regime-aware path.
6. Prints summary metrics and points to the cross-language parity contract.

Reference: see [`README.md`](../../README.md) for the full feature list and [`CHANGELOG.md`](../../CHANGELOG.md) for what shipped in each version.

In [ ]:
# Cell 2 — generate a small synthetic OHLC CSV with the bundled GBM
# generator. Output goes to data/SYNTHETIC.csv in the same shape as the
# Binance downloader: header = time,open,high,low,close.
#
# We chdir to the repo root so the generator's default output path
# (data/SYNTHETIC.csv) lands in the expected place regardless of which
# directory Jupyter was launched from.
import os, subprocess, sys, pathlib

REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
print('repo root:', REPO_ROOT)

# Prefer the bundled real-market sample if present; fall back to synthetic.
candidate = REPO_ROOT / 'data_SOLUSDT_1h.csv'
if candidate.exists():
    CSV_PATH = str(candidate)
    print(f'using bundled sample: {CSV_PATH}')
else:
    out = REPO_ROOT / 'data' / 'SYNTHETIC.csv'
    out.parent.mkdir(exist_ok=True)
    subprocess.check_call([sys.executable, 'gen_synthetic.py'])
    CSV_PATH = str(out)
    print(f'generated synthetic: {CSV_PATH}')

# The engine reads BT_CSV at import time, so set it before importing.
os.environ['BT_CSV'] = CSV_PATH

In [ ]:
# Cell 3 — import the package. Should print nothing and complete fast.
import backtester as bt
print('backtester imported OK — default CSV resolved to:', bt.CSV_FILE)

## In-Sample / Out-of-Sample / Walk-Forward

The engine splits the data into an **In-Sample (IS)** training window and an
**Out-of-Sample (OOS)** holdout, optimises the look-back parameter on IS, then
scores on OOS. **Walk-Forward Optimisation (WFO)** repeats this stride-by-stride
across the timeline so the OOS score is averaged over many fresh, never-seen
windows rather than one lucky split. Optional realism overlays (fee shock,
slippage shock, indicator-variance, entry-drift, news-candle injection) are
scored against every WFO window and reported next to the clean baseline.

In [ ]:
# Cell 5 — run the default pipeline. The engine ships with a built-in
# create_raw_signals (an EMA(20) vs EMA(lb) crossover with a one-bar
# shift to avoid look-ahead) wired to the IS/OOS plumbing and the
# pre-computed EMA columns. We keep that as the demo strategy.
#
# To plug your own in:
#
#   def my_strategy(df, lb):
#       # use df['EMA_20'].shift(1) etc. — the engine pre-computes these.
#       return np.zeros(len(df), dtype=np.int8)
#   bt.create_raw_signals = my_strategy
#
# We shrink the windows so the notebook runs in under a minute.
bt.BACKTEST_CANDLES = 5000
bt.OOS_CANDLES = 5000
bt.PRINT_EQUITY_CURVE = False  # we plot ourselves below
bt.USE_MONTE_CARLO = False     # MC adds 1000 reshuffles — skip in the demo
bt.USE_WFO = False             # one IS/OOS slice is enough for the tour
bt.main()

In [ ]:
# Cell 6 — plot the equity curve from the exported trade ledger. The
# engine writes one row per closed trade to bt.EXPORT_PATH (default
# trade_list.csv); we cumsum the per-trade PnL into an equity series.
import pandas as pd, matplotlib.pyplot as plt

ledger = pd.read_csv(bt.EXPORT_PATH)
print(f'ledger rows: {len(ledger)}; columns: {list(ledger.columns)}')
if 'pnl' in ledger.columns and len(ledger):
    eq = ledger['pnl'].cumsum() + bt.ACCOUNT_SIZE
    plt.figure(figsize=(10, 4))
    plt.plot(eq.values)
    plt.title('Equity curve from trade ledger')
    plt.xlabel('trade #')
    plt.ylabel('equity (account currency)')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
else:
    print('No trades produced — likely the look-back range did not hit on this slice.')

## Robustness suite

Each WFO window is re-scored with five overlays:

| Overlay | What it does |
|---|---|
| `ENTRY_DRIFT` | Shift entries one bar forward. |
| `FEE_SHOCK` | 2× fees on every trade. |
| `SLIPPAGE_SHOCK` | 3× slippage on entry and exit. |
| `INDICATOR_VARIANCE` | ±1 perturbation on the selected look-back. |
| `NEWS_CANDLES_INJECTION` | Synthetic high-vol wicks every 500–1000 bars. |

An edge that survives all five with an OOS Sharpe still positive is the
kind of result that justifies a paper-trade allocation; an edge that
evaporates when fees double was probably curve-fitting to a quiet stretch.

In [ ]:
# Cell 8 — toggle USE_REGIME_SEG = True and re-run to show the regime
# segmentation surface. The regime engine fits a per-regime look-back
# (Uptrend / Downtrend / Ranging by default) and rotates inside each
# WFO window. We enable PRINT_EQUITY_CURVE so the engine builds the
# stitched IS+OOS equity series for the regime path.
bt.USE_REGIME_SEG = True
bt.USE_WFO = False
bt.PRINT_EQUITY_CURVE = True
bt.main()

## Cross-language parity

This Python engine has a 1-to-1 Rust port at
[`DaruFinance/quant-research-framework-rs`](https://github.com/DaruFinance/quant-research-framework-rs).

Four parity scripts in the Rust repo (`tools/parity_check.py`,
`parity_regime.py`, `parity_forex.py`, `parity_ledger.py`) run both
engines on the same CSV and assert the printed metric ledger agrees to
$10^{-3}$ relative tolerance. Today's status: **210 / 210 metric points**
across the three published surfaces (default + regime+WFO + forex), max
observed deviation below $5\!\times\!10^{-5}$. The CI workflow at
`.github/workflows/parity.yml` enforces this on every PR.

In [ ]:
# Cell 10 — print final summary metrics from the trade ledger.
ledger = pd.read_csv(bt.EXPORT_PATH)
if len(ledger) and 'pnl' in ledger.columns:
    pnl = ledger['pnl']
    summary = {
        'trades': int(len(pnl)),
        'total_pnl': float(pnl.sum()),
        'win_rate': float((pnl > 0).mean()),
        'avg_pnl': float(pnl.mean()),
        'max_drawdown': float(((pnl.cumsum().cummax() - pnl.cumsum())).max()),
    }
    for k, v in summary.items():
        print(f'{k:>14}: {v:.4f}' if isinstance(v, float) else f'{k:>14}: {v}')
else:
    print('Ledger is empty; nothing to summarise.')